# Kleo2iOS — Colab/Linux preparation

This notebook downloads your ZIP directly into Google Drive, safely extracts the `.nup` from it, performs the Linux-safe stages from the pinned NXNative builder, then uploads a private handoff archive and queues the Kleo2iOS-only GitHub Actions workflow.

**Do not use this notebook to upload console keys.** A NUP container does not require prod.keys. The ZIP, extracted NUP and handoff archive stay in your private Drive; the service account only needs Viewer access to the handoff folder.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, subprocess, sys, os, json, hashlib, uuid, datetime, getpass, base64, requests, zipfile, io
from urllib.parse import urlparse

drive.mount('/content/drive')

# Authorized source magnet supplied by the user. It is kept in this notebook cell,
# not printed in output. The user can choose a direct ZIP URL instead.
TORRENT_MAGNET = 'magnet:?xt=urn:btih:3888c56219256787bfe7b3e08a69f71632d927f9&xt=urn:btmh:12203041aed70b89adbd9f676addd2aa63560c4be8752a2e9af62100af2643d583ab&dn=Donkey%20Kong%20Bananza%20+%20Update%203.2.0%20+%20DLC&tr=udp://tracker.opentrackr.org:1337/announce&tr=udp://open.stealth.si:80/announce&tr=udp://tracker.torrent.eu.org:451/announce'
SOURCE_MODE = (input('Source: [T]orrent magnet (default) or [U]RL ZIP? ').strip().lower() or 't')
assert SOURCE_MODE in ('t','torrent','u','url'), 'Choose T for torrent or U for direct ZIP URL'
DRIVE_FOLDER_ID = input('Google Drive folder ID shared with the Actions service account: ').strip()
assert DRIVE_FOLDER_ID, 'Drive folder ID is required'

INPUT_DIR = Path('/content/drive/MyDrive/Kleo2iOS_Input')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
NUP_PATH = INPUT_DIR / 'game.nup'
if NUP_PATH.exists(): NUP_PATH.unlink()

if SOURCE_MODE in ('t','torrent'):
    TORRENT_DIR = INPUT_DIR / 'torrent-download'
    TORRENT_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y aria2'],check=True)
    print('Starting BitTorrent download directly into Google Drive. aria2 will show live progress; do not close this Colab session.')
    cmd = ['aria2c','--enable-dht=true','--bt-enable-lpd=true','--bt-max-peers=80','--seed-time=0','--summary-interval=10','--console-log-level=notice','--continue=true','--file-allocation=none','--dir',str(TORRENT_DIR),TORRENT_MAGNET]
    subprocess.run(cmd,check=True)
    candidates = [p for p in TORRENT_DIR.rglob('*') if p.is_file() and p.suffix.lower()=='.nup']
    assert len(candidates)==1, 'Expected exactly one .nup from torrent; found: '+str([str(p.relative_to(TORRENT_DIR)) for p in candidates])
    shutil.copy2(candidates[0],NUP_PATH)
    print('Torrent download complete:', candidates[0].name, '| bytes:', candidates[0].stat().st_size)
else:
    DOWNLOAD_URL = getpass.getpass('Paste the direct ZIP download link: ').strip()
    assert DOWNLOAD_URL.startswith(('https://','http://')), 'A valid HTTP(S) download URL is required'
    ZIP_PATH = INPUT_DIR / 'source-game.zip'
    PART_PATH = INPUT_DIR / 'source-game.zip.part'
    for p in (ZIP_PATH,PART_PATH):
        if p.exists(): p.unlink()
    print('Downloading ZIP directly to Google Drive...')
    with requests.get(DOWNLOAD_URL,stream=True,allow_redirects=True,timeout=(30,180),headers={'User-Agent':'Mozilla/5.0 Kleo2iOS-Colab/1.0'}) as response:
        print('HTTP:',response.status_code,'| content-type:',response.headers.get('Content-Type','unknown'),'| final host:',urlparse(response.url).hostname)
        response.raise_for_status()
        first=b''
        with PART_PATH.open('wb') as out:
            for chunk in response.iter_content(chunk_size=16*1024*1024):
                if not chunk: continue
                if not first: first=chunk[:8]
                out.write(chunk)
        assert first.startswith((b'PK\\x03\\x04',b'PK\\x05\\x06',b'PK\\x07\\x08')), f'Server did not return ZIP bytes: {first!r}'
        assert PART_PATH.stat().st_size>0,'Downloaded ZIP is empty'
        PART_PATH.replace(ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        candidates=[i for i in archive.infolist() if not i.is_dir() and i.filename.lower().endswith('.nup')]
        assert len(candidates)==1,'Expected exactly one .nup in ZIP; found: '+str([i.filename for i in candidates])
        member=candidates[0]
        assert member.file_size>0 and not (member.flag_bits&0x1),'NUP entry is empty or encrypted'
        written=0
        with archive.open(member,'r') as src,NUP_PATH.open('wb') as dst:
            while True:
                block=src.read(16*1024*1024)
                if not block: break
                dst.write(block); written+=len(block)
        assert written==member.file_size,'Extracted NUP size does not match ZIP metadata'

assert NUP_PATH.is_file() and NUP_PATH.stat().st_size>0 and NUP_PATH.suffix.lower()=='.nup'
sha=hashlib.sha256()
with NUP_PATH.open('rb') as f:
    for block in iter(lambda:f.read(16*1024*1024),b''): sha.update(block)
print('Input NUP ready in Drive:',NUP_PATH,'| bytes:',NUP_PATH.stat().st_size,'| sha256:',sha.hexdigest())


In [ ]:
from pathlib import Path
NX2APPLE = Path('/content/NX2Apple')
if NX2APPLE.exists(): shutil.rmtree(NX2APPLE)
subprocess.run(['git','clone','--depth','1','--branch','v1.3.0-rc10','https://github.com/LNPANDY/NX2Apple.git',str(NX2APPLE)],check=True)
subprocess.run(['python','tools/engine.py','sync'],cwd=NX2APPLE,check=True)
print('NX2Apple pin:', (NX2APPLE/'.upstream-ref').read_text().strip())
assert sys.version_info >= (3,12), f'Python 3.12+ required; got {sys.version}'
subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y zstd'],check=True)
subprocess.run([sys.executable,'-m','pip','install','lz4','capstone>=5,<6','fonttools','zbic','backports.zstd','requests','google-api-python-client','google-auth'],check=True)

In [ ]:
PREP_SCRIPT = Path('/content/prepare_linux.py')
raw_url = 'https://raw.githubusercontent.com/joao9vegeto-stack/KintTany/Kleo2iOS/colab/prepare_linux.py'
r = requests.get(raw_url, timeout=60); r.raise_for_status(); PREP_SCRIPT.write_bytes(r.content)
WORK_DIR = Path('/content/kleo2ios-work')
if WORK_DIR.exists(): shutil.rmtree(WORK_DIR)
subprocess.run([sys.executable,str(PREP_SCRIPT),'--engine-root',str(NX2APPLE/'engine'),'--nx2apple-root',str(NX2APPLE),'--nup',str(NUP_PATH),'--work-dir',str(WORK_DIR)],check=True)
PREP = json.loads((WORK_DIR/'kleo-linux-prep.json').read_text())
print('Prepared title:', PREP['title_id'], 'package:', PREP['package_version'])

In [ ]:
HANDOFF_DIR = Path('/content/kleo2ios-handoff')
if HANDOFF_DIR.exists(): shutil.rmtree(HANDOFF_DIR)
HANDOFF_DIR.mkdir(parents=True)
shutil.copy2(NUP_PATH, HANDOFF_DIR/'game.nup')
shutil.copytree(WORK_DIR, HANDOFF_DIR/'work')
manifest = {'format':1,'title_id':PREP['title_id'],'package_version':PREP['package_version'],'engine_pin':PREP['engine_pin'],'nx2apple_tag':PREP['nx2apple_tag'],'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
(HANDOFF_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
ARCHIVE = Path('/content/kleo2ios-handoff.tar.zst')
if ARCHIVE.exists(): ARCHIVE.unlink()
p1 = subprocess.Popen(['tar','-cf','-','-C',str(HANDOFF_DIR),'game.nup','work','manifest.json'],stdout=subprocess.PIPE)
p2 = subprocess.run(['zstd','-1','-o',str(ARCHIVE)],stdin=p1.stdout,check=False)
p1.stdout.close()
assert p1.wait() == 0 and p2.returncode == 0, 'Could not create zstd handoff archive'
sha = hashlib.sha256()
with ARCHIVE.open('rb') as f:
    for block in iter(lambda:f.read(16*1024*1024),b''): sha.update(block)
ARCHIVE_SHA256 = sha.hexdigest(); ARCHIVE_SIZE = ARCHIVE.stat().st_size
print('Handoff archive:', ARCHIVE, 'size:', ARCHIVE_SIZE, 'sha256:', ARCHIVE_SHA256)

In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
auth.authenticate_user()
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive_api = build('drive','v3',credentials=creds,cache_discovery=False)
request_id = str(uuid.uuid4())
metadata = {'name':f'Kleo2iOS-{PREP["title_id"]}-{request_id}.tar.zst','parents':[DRIVE_FOLDER_ID]}
media = MediaFileUpload(str(ARCHIVE),mimetype='application/zstd',chunksize=64*1024*1024,resumable=True)
uploaded = drive_api.files().create(body=metadata,media_body=media,fields='id,name,size').execute()
DRIVE_FILE_ID = uploaded['id']
print('Private Drive upload complete:', uploaded)

In [ ]:
github_token = getpass.getpass('Fine-grained GitHub token (Contents: Read and write for KintTany): ')
handoff = {'format':1,'request_id':request_id,'drive_file_id':DRIVE_FILE_ID,'archive_sha256':ARCHIVE_SHA256,'archive_size':ARCHIVE_SIZE,'title_id':PREP['title_id'],'package_version':PREP['package_version'],'engine_pin':PREP['engine_pin'],'nx2apple_tag':PREP['nx2apple_tag'],'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
api_url = 'https://api.github.com/repos/joao9vegeto-stack/KintTany/contents/handoff.json'
headers = {'Authorization':f'Bearer {github_token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
existing = requests.get(api_url,headers=headers,params={'ref':'Kleo2iOS'},timeout=30)
sha_existing = existing.json().get('sha') if existing.status_code == 200 else None
if existing.status_code not in (200,404): raise RuntimeError(existing.text)
payload = {'message':f'Queue Kleo2iOS build {PREP["title_id"]} {request_id}','content':base64.b64encode((json.dumps(handoff,indent=2)+'\n').encode()).decode(),'branch':'Kleo2iOS'}
if sha_existing: payload['sha'] = sha_existing
created = requests.put(api_url,headers=headers,json=payload,timeout=60); created.raise_for_status()
print('Handoff manifest committed to Kleo2iOS. This commit triggers only the Kleo2iOS workflow.')
print('Commit:', created.json()['commit']['html_url'])
print('Actions:', 'https://github.com/joao9vegeto-stack/KintTany/actions')
github_token = None